In [1]:
#Task 4
# İlk eğitim adımları. Modeli rastgele başlat ve ilk loss'u ölç, ln(50257)
# ≈ 10.8 ile karşılaştır. Neden bu değere yakın çıkmalı, kendi
# cümlelerinle yaz. Sonra tek bir batch'i ezberlet: aynı batch'te birkaç
# yüz adım eğit, loss sıfıra yaklaşmalı. Bu test neden işe yarıyor?
# (0:45:50 - 1:02:00)

In [2]:
#imports
import tiktoken
from dataclasses import dataclass
import torch
import torch.nn as nn
from torch.nn import functional as F
import math

In [3]:
class CausalSelfAttention(nn.Module):
    def __init__(self,config):
        super().__init__()
        assert config.n_embd % config.n_head == 0
        #batchwisely k q v projections for all heads
        self.c_attn = nn.Linear(config.n_embd, 3 * config.n_embd)
        #output projection
        self.c_proj = nn.Linear(config.n_embd, config.n_embd)
        #regularization
        self.n_head = config.n_head
        self.n_embd = config.n_embd
        #mask
        self.register_buffer("bias", torch.tril(torch.ones(config.block_size, config.block_size))
            .view(1, 1, config.block_size, config.block_size))


    def forward(self, x):
        B, T, C = x.size() #batch size, sequence length, embedding dims

        qkv = self.c_attn(x)
        q, k, v = qkv.split(self.n_embd, dim = 2)
        k = k.view(B,T, self.n_head, C // self.n_head).transpose(1,2) #B, T, nh, hs to B, nh, T, hs
        q = q.view(B,T, self.n_head, C // self.n_head).transpose(1,2) #B, T, nh, hs to B, nh, T, hs
        v = v.view(B,T, self.n_head, C // self.n_head).transpose(1,2) #B, T, nh, hs to B, nh, T, hs
        #attention
        att = (q @ k.transpose(-2,-1)) * (1.0 / math.sqrt(k.size(-1)))
        att = att.masked_fill(self.bias[:,:,:T,:T] == 0, float("-inf"))
        att = F.softmax(att, dim=-1)

        y = att @ v #(B, nh, T, T) x (B, nh, T, hs) = (B, nh, T, hs)
        #taking all head outputs side by side
        y = y.transpose(1,2).contiguous().view(B,T,C) 
        #output projection
        y = self.c_proj(y)
        return y

In [4]:
class MLP(nn.Module):
    def __init__(self,config):
        super().__init__()
        self.c_fc = nn.Linear(config.n_embd, 4 * config.n_embd)
        self.gelu = nn.GELU(approximate = "tanh") #nonlinearity
        self.c_proj = nn.Linear(4 * config.n_embd, config.n_embd)

    def forward(self,x):
        x = self.c_fc(x)
        x = self.gelu(x)
        x = self.c_proj(x)
        return x
        

In [5]:
class Block(nn.Module):
    def __init__(self,config):
        super().__init__()
        self.ln_1 = nn.LayerNorm(config.n_embd)
        self.attn = CausalSelfAttention(config)
        self.ln_2 = nn.LayerNorm(config.n_embd)
        self.mlp = MLP(config)

    def forward(self,x):
        x = x + self.attn(self.ln_1(x))
        x = x + self.mlp(self.ln_2(x))
        return x 

In [6]:
@dataclass
class GPTConfig:
    block_size: int= 1024
    vocab_size: int = 50257
    n_layer: int= 12 
    n_head: int= 12 
    n_embd: int= 768

class GPT(nn.Module):
    def __init__(self,config):
        super().__init__()
        self.config = config

        self.transformer = nn.ModuleDict(dict(
                    wte = nn.Embedding(config.vocab_size, config.n_embd), #token embds
                    wpe = nn.Embedding(config.block_size, config.n_embd), #positional embds
                    h = nn.ModuleList([Block(config) for _ in range(config.n_layer)]), #hidden blocks
                    ln_f = nn.LayerNorm(config.n_embd),)) #final layernorm
        
        self.lm_head = nn.Linear(config.n_embd, config.vocab_size, bias=False) #final classifier  
   
    def forward(self, idx, targets=None):
        # idx is of shape (B, T)
        B, T = idx.size()
        assert T <= self.config.block_size, f"Cannot forward sequence of length {T}, block size is only {self.config.block_size}"
        #first embeddings token and positional
        pos = torch.arange(0, T, dtype=torch.long, device=idx.device) # shape (T)
        pos_emb = self.transformer.wpe(pos) # position embeddings of shape (T, n_embd)
        tok_emb = self.transformer.wte(idx) # token embeddings of shape (B, T, n_embd)
        x = tok_emb + pos_emb
        #going for blocks
        for block in self.transformer.h:
            x = block(x)
        #final layernorm and the classifier
        x = self.transformer.ln_f(x)
        logits = self.lm_head(x) # (B, T, vocab_size)
        loss = None

        if targets is not None:
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)),targets.view(-1))
        return logits, loss
    
    @classmethod
    def from_pretrained(cls, model_type):
        assert model_type in {'gpt2', 'gpt2-medium', 'gpt2-large', 'gpt2-xl'}
        from transformers import GPT2LMHeadModel
        print("loading weights from pretrained gpt: %s" % model_type)

        # n_layer, n_head and n_embd are determined from model_type
        config_args = {
            'gpt2':         dict(n_layer=12, n_head=12, n_embd=768),  # 124M params
            'gpt2-medium':  dict(n_layer=24, n_head=16, n_embd=1024), # 350M params
            'gpt2-large':   dict(n_layer=36, n_head=20, n_embd=1280), # 774M params
            'gpt2-xl':      dict(n_layer=48, n_head=25, n_embd=1600), # 1558M params
        }[model_type]
        config_args['vocab_size'] = 50257 # always 50257 for GPT model checkpoints
        config_args['block_size'] = 1024 # always 1024 for GPT model checkpoints
        # create a from-scratch initialized minGPT model
        config = GPTConfig(**config_args)
        model = GPT(config)
        sd = model.state_dict()
        sd_keys = sd.keys()
        sd_keys = [k for k in sd_keys if not k.endswith('.attn.bias')] # discard this mask / buffer, not a param

        # init a huggingface/transformers model
        model_hf = GPT2LMHeadModel.from_pretrained(model_type)
        sd_hf = model_hf.state_dict()

        # copy while ensuring all of the parameters are aligned and match in names and shapes
        sd_keys_hf = sd_hf.keys()
        sd_keys_hf = [k for k in sd_keys_hf if not k.endswith('.attn.masked_bias')] # ignore these, just a buffer
        sd_keys_hf = [k for k in sd_keys_hf if not k.endswith('.attn.bias')] # same, just the mask (buffer)
        transposed = ['attn.c_attn.weight', 'attn.c_proj.weight', 'mlp.c_fc.weight', 'mlp.c_proj.weight']
        # basically the openai checkpoints use a "Conv1D" module, but we only want to use a vanilla Linear
        # this means that we have to transpose these weights when we import them
        assert len(sd_keys_hf) == len(sd_keys), f"mismatched keys: {len(sd_keys_hf)} != {len(sd_keys)}"
        for k in sd_keys_hf:
            if any(k.endswith(w) for w in transposed):
                # special treatment for the Conv1D weights we need to transpose
                assert sd_hf[k].shape[::-1] == sd[k].shape
                with torch.no_grad():
                    sd[k].copy_(sd_hf[k].t())
            else:
                # vanilla copy over the other parameters
                assert sd_hf[k].shape == sd[k].shape
                with torch.no_grad():
                    sd[k].copy_(sd_hf[k])

        return model

In [7]:
enc = tiktoken.get_encoding("gpt2")
with open('input.txt', 'r') as f:
    text = f.read()
text = text[:1000]
tokens = enc.encode(text)
B, T = 4, 32 
buf = torch.tensor(tokens[:B*T + 1])
x = buf[:-1].view(B,T)
y = buf[1:].view(B,T)

In [8]:
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
model = GPT(GPTConfig())
model.eval()
model.to(device)
x = x.to(device)
y = y.to(device)

In [9]:
logits, loss = model(x,y)

In [10]:
loss

tensor(11.0334, device='mps:0', grad_fn=<NllLossBackward0>)

In [11]:
#lets optimize
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4)
for i in range(100):
    optimizer.zero_grad()
    logits, loss = model(x,y)
    loss.backward()
    optimizer.step()
    print(f' step {i}, loss is {loss.item()}') 

 step 0, loss is 11.03337287902832
 step 1, loss is 6.599831581115723
 step 2, loss is 4.2411956787109375
 step 3, loss is 2.587937355041504
 step 4, loss is 1.4687776565551758
 step 5, loss is 0.7966665029525757
 step 6, loss is 0.432366281747818
 step 7, loss is 0.2547517418861389
 step 8, loss is 0.1602778136730194
 step 9, loss is 0.10408568382263184
 step 10, loss is 0.0725327879190445
 step 11, loss is 0.054918862879276276
 step 12, loss is 0.04294048994779587
 step 13, loss is 0.034268882125616074
 step 14, loss is 0.028873542323708534
 step 15, loss is 0.025203777477145195
 step 16, loss is 0.0218767449259758
 step 17, loss is 0.018833961337804794
 step 18, loss is 0.01643257960677147
 step 19, loss is 0.014652656391263008
 step 20, loss is 0.013263722881674767
 step 21, loss is 0.01206594705581665
 step 22, loss is 0.010970277711749077
 step 23, loss is 0.00997831579297781
 step 24, loss is 0.009111976251006126
 step 25, loss is 0.008371803909540176
 step 26, loss is 0.0077403